# Practice: Audit a Corpus You Intend to Publish On

**Computational Text Analysis Pack · Module 22 — Distant Reading and Its Critics**

*Technique:* The five-question absence audit, applied and written up

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/11-distant-reading-and-its-critics/22d_practice_absence_audit.ipynb)

Data: `colonial_texts.csv`, `masakhanews.csv`, `african_legal.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['colonial_texts.csv', 'masakhanews.csv', 'african_legal.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df_colonial = pd.read_csv('colonial_texts.csv')
df       = pd.read_csv('masakhanews.csv')
df_legal = pd.read_csv('african_legal.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df_colonial has {len(df_colonial):,} rows and {df_colonial.shape[1]} columns.')

## The question

> **Before you publish, what does your corpus not contain — and what claims does
> that rule out?**

Module 22c gave five questions. This notebook answers them for all three corpora
in this course, and produces the paragraph you would put in a data-availability
statement.

### How to use this notebook

A worked case, not a tutorial. Run it once for the shape of the argument, then
change the question at the top to one of your own.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

In [ ]:
import numpy as np
CORPORA = {"colonial_texts": df_colonial, "masakhanews": df, "african_legal": df_legal}

print("Q1. WHO PRODUCED THESE TEXTS?\n")
for name, frame in CORPORA.items():
    top = frame["author"].value_counts() if "author" in frame.columns else pd.Series(dtype=int)
    if len(top):
        conc = top.head(max(1, len(top)//10)).sum()/len(frame)
        print(f"{name:<18}{len(top):>4} named authors; top 10% hold {conc:.0%} of rows")
        print(f"{'':18}most represented: {str(top.index[0])[:48]}")
    else:
        print(f"{name:<18}authorship not recorded — a limit worth stating, not a gap to ignore")
    print()

In [ ]:
print("Q2. HOW DID THESE TEXTS SURVIVE TO BE DIGITISED?\n")
notes = {
 "colonial_texts": "Gutenberg keyword search for 'africa'. Selection happened twice: "
                   "what was printed pre-1929, and what volunteers chose to digitise.",
 "masakhanews":    "Outlets with an online presence and a digitised archive. "
                   "Languages without a digital news industry are absent by construction.",
 "african_legal":  "Institutional publication. Only bodies that publish openly appear.",
}
for k, v in notes.items():
    print(f"{k}:\n   {v}\n")

print("Q3. WHAT DOES ONE ROW REPRESENT?\n")
for name, frame in CORPORA.items():
    unit = "a ~220-word passage" if "passage_no" in frame.columns else "one news article"
    n_words = frame["n_words"].median() if "n_words" in frame.columns else float("nan")
    print(f"{name:<18}{unit:<24}median {n_words:,.0f} words, {len(frame):,} rows")

In [ ]:
print("\nQ4. WHICH 'AUTHORS' ARE ACTUALLY INSTITUTIONS?\n")
INSTITUTIONAL = ["various", "unknown", "administration", "society", "company",
                 "office", "committee", "anonymous", "department"]
for name, frame in CORPORA.items():
    if "author" not in frame.columns or frame["author"].dropna().empty: continue
    sus = [a for a in frame["author"].dropna().unique()
           if any(k in str(a).lower() for k in INSTITUTIONAL)]
    n = frame[frame["author"].isin(sus)].shape[0]
    print(f"{name:<18}{len(sus):>3} non-person values covering {n:,} rows "
          f"({n/len(frame):.1%})")
    for s in sus[:4]:
        print(f"{'':20}{s}")
print()
print("Any per-author analysis silently treats these as people. Module 9's")
print("stylometry filtered them out by requiring two works — by luck, not design.")

In [ ]:
print("Q5. WHAT CLAIM WOULD THIS CORPUS REFUTE?\n")
CAN_REFUTE = {
 "colonial_texts": "That pre-1929 anglophone writing about Africa did NOT use a "
                   "particular vocabulary. It cannot refute anything about African writing.",
 "masakhanews":    "That a topic is absent from online African news in these 16 "
                   "languages. It cannot refute anything about speech, or about "
                   "languages without a digital press.",
 "african_legal":  "That an institution did not publish a given formulation. It "
                   "cannot speak to unpublished or internal documents.",
}
for k, v in CAN_REFUTE.items():
    print(f"{k}:\n   {v}\n")

fig, ax = plt.subplots(figsize=(7.5, 3.6))
langs = df["language"].value_counts()
ax.bar(range(len(langs)), langs.values, color="steelblue")
ax.set_xticks(range(len(langs))); ax.set_xticklabels(langs.index, rotation=60,
                                                     ha="right", fontsize=7)
ax.set_ylabel("documents")
ax.set_title(f"{len(langs)} languages present, out of 2,000+ spoken in Africa")
plt.tight_layout()

## The paragraph this produces

> The corpus comprises [N] documents drawn from [source], collected [when] by
> [method]. One row is [unit]. Authorship is recorded for [X]% of rows, of which
> [Y] values are institutional rather than personal. Selection occurred at
> [stages], so the corpus represents [what] rather than [what it might be
> mistaken for]. Findings therefore support claims about [scope] and cannot
> support claims about [excluded scope].

Write that before the analysis. A limits section drafted first tells you which
findings you are permitted to look for — which is faster than discovering it in
review, and more honest than discovering it and not saying.

### 🔧 Your turn 1

Run all five questions against the corpus you plan to publish on, and write the paragraph. Which question was hardest to answer?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Question 2 — how the texts survived to be digitised — is almost always the
hardest, because the answer is usually not documented anywhere and requires
tracing an archive's own history. That difficulty is Bode's argument in
practical form: the bibliographic and archival scholarship is not preliminary to
computational literary study, it is part of it, and a project that cannot answer
question 2 does not yet know what it is measuring.

</details>